**Instituto Brasileiro de Ensino, Desenvolvimento e Pesquisa**

Programa de Pós-Graduação em Administração Pública  
**Mestrado Profissional em Administração Pública - Ciência de Dados e Inteligência Artificial**

**Pesquisa de Dissertação**  
**Aluno**: Fabricio Fernandes Santana  
**Orientador**: Prof. Dr. Marcelo Rodrigo de Souza Pita

## Análise da base de discursos da 56ª Legislatura do Senado Federal

Este notebook realiza uma auditoria reprodutível do corpus de discursos da 56ª Legislatura (01/02/2019 a 31/01/2023). Além da análise exploratória tradicional, investiga fatores que afetam uma solução de *Retrieval-Augmented Generation* (RAG): cobertura textual, qualidade dos documentos, duplicidade, distribuição de comprimentos, estimativa de tokens e chunks, qualidade dos metadados, desbalanceamentos e variação lexical no tempo.

A análise procura responder:
- qual é a cobertura e a integridade do corpus;
- como os discursos se distribuem no tempo e entre autores, partidos e UFs;
- quais propriedades textuais influenciam indexação, chunking e recuperação;
- onde há risco de redundância, ausência de contexto ou viés de representação; e 
- quais decisões de preparação devem anteceder a construção do RAG.


### 1. Configurar notebook



Execute a célula abaixo, se você clonou o [respositório com todos os arquivos](https://github.com/fabriciosantana/mcdia/tree/main/13-dissertacao). Caso esteja no Google Colab, verique [aqui](https://github.com/fabriciosantana/mcdia/blob/main/13-dissertacao/notebooks/requirements.txt) as dependência que você deve instalar

In [ ]:
!python -m pip install -r requirements.txt

In [1]:
from __future__ import annotations

import hashlib
import math
import re
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from huggingface_hub import hf_hub_download
from scipy.spatial.distance import jensenshannon
from sklearn.feature_extraction.text import CountVectorizer

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid', context='notebook', palette='colorblind')
plt.rcParams.update({'figure.figsize': (11, 5.5), 'figure.dpi': 120})
RANDOM_STATE = 42

## 2. Aquisição reprodutível

A célula seguinte procura o arquivo em caminhos compatíveis com a execução a partir da raiz do repositório ou do diretório do notebook. Na ausência do arquivo, `hf_hub_download` recupera exatamente o Parquet publicado. O hash esperado documenta a versão analisada.

In [ ]:
HF_REPO_ID = 'fabriciosantana/discursos-senado-legislatura-56'
HF_FILENAME = 'data/full/discursos_2019-02-01_2023-01-31.parquet'
HF_REVISION = 'v1.1.1'
EXPECTED_SHA256 = 'e09cfc4793e5394be440906320c7d3008cda5a52b90bbc85bf47446362406af1'
ARQUIVO = 'discursos_2019-02-01_2023-01-31.parquet'

candidatos = [
    Path.cwd() / '13-dissertacao' / 'dados' / ARQUIVO,
    Path.cwd().parent / 'dados' / ARQUIVO,
    Path.cwd() / 'dados' / ARQUIVO,
]
caminho_local = next((p.resolve() for p in candidatos if p.exists()), None)

if caminho_local is None:
    caminho_local = Path(hf_hub_download(
        repo_id=HF_REPO_ID,
        repo_type='dataset',
        filename=HF_FILENAME,
        revision=HF_REVISION,
    ))
    origem = f'Hugging Face ({HF_REPO_ID}@{HF_REVISION})'
else:
    origem = 'arquivo local'

sha256 = hashlib.sha256(caminho_local.read_bytes()).hexdigest()
print(f'Origem: {origem}')
print(f'Arquivo: {caminho_local}')
print(f'SHA-256: {sha256}')
assert sha256 == EXPECTED_SHA256, 'O arquivo não corresponde à versão documentada v1.1.1.'

In [ ]:
df_raw = pd.read_parquet(caminho_local)
print(f'{len(df_raw):,} registros × {df_raw.shape[1]} colunas')
display(df_raw.head(3))

## 3. Estrutura, integridade e completude

Primeiro auditamos esquema, tipos, valores ausentes, chaves e intervalo temporal. Essa etapa separa problemas estruturais de ausências esperadas na fonte.

In [ ]:
def contar_unicos(serie: pd.Series) -> int:
    # Colunas aninhadas podem conter arrays/listas, que não são diretamente hashable.
    if serie.dtype == 'object':
        return serie.dropna().map(repr).nunique()
    return serie.nunique(dropna=True)

visao_colunas = (
    pd.DataFrame({
        'tipo': df_raw.dtypes.astype(str),
        'preenchidos': df_raw.notna().sum(),
        'ausentes': df_raw.isna().sum(),
        'unicos': pd.Series({c: contar_unicos(df_raw[c]) for c in df_raw.columns}),
    })
    .assign(ausentes_pct=lambda x: (100 * x['ausentes'] / len(df_raw)).round(2))
    .sort_values(['ausentes_pct', 'unicos'], ascending=[False, True])
)
display(visao_colunas)

In [ ]:
df = df_raw.copy()
df['Data'] = pd.to_datetime(df['Data'], errors='coerce')
for coluna in ['Resumo', 'Indexacao', 'TextoDiscursoIntegral', 'NomeAutor', 'Partido', 'UF']:
    df[coluna] = df[coluna].fillna('').astype(str).str.strip()

integridade = pd.Series({
    'registros': len(df),
    'colunas': df.shape[1],
    'códigos únicos': df['CodigoPronunciamento'].nunique(dropna=True),
    'códigos duplicados': df['CodigoPronunciamento'].duplicated(keep=False).sum(),
    'datas inválidas': df['Data'].isna().sum(),
    'data mínima': df['Data'].min(),
    'data máxima': df['Data'].max(),
    'autores': df.loc[df['NomeAutor'].ne(''), 'NomeAutor'].nunique(),
    'partidos': df.loc[df['Partido'].ne(''), 'Partido'].nunique(),
    'UFs': df.loc[df['UF'].ne(''), 'UF'].nunique(),
}, name='valor')
display(integridade.to_frame())

In [ ]:
faltantes_plot = visao_colunas.query('ausentes > 0').sort_values('ausentes_pct').tail(15)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=faltantes_plot.reset_index(), x='ausentes_pct', y='index', ax=ax)
ax.set(title='Colunas com maior proporção de valores ausentes', xlabel='Ausentes (%)', ylabel='Coluna')
ax.bar_label(ax.containers[0], fmt='%.1f%%', padding=3)
plt.tight_layout()

### 3.1 Cobertura dos textos

Para o RAG, não basta contar linhas. É necessário distinguir texto integral, resumo substituto e registro sem conteúdo recuperável. A coluna `fonte_documento` conserva essa proveniência para filtros e avaliações posteriores.

In [ ]:
tem_integral = df['TextoDiscursoIntegral'].ne('')
tem_resumo = df['Resumo'].ne('')
df['fonte_documento'] = np.select(
    [tem_integral, ~tem_integral & tem_resumo],
    ['texto_integral', 'resumo_fallback'],
    default='indisponível',
)
df['documento_rag'] = np.where(tem_integral, df['TextoDiscursoIntegral'], df['Resumo'])

cobertura = df['fonte_documento'].value_counts().rename_axis('fonte').to_frame('registros')
cobertura['percentual'] = (100 * cobertura['registros'] / len(df)).round(2)
display(cobertura)
display(pd.crosstab(df['status'], df['fonte_documento'], margins=True))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.countplot(data=df, x='fonte_documento', order=['texto_integral', 'resumo_fallback', 'indisponível'], ax=axes[0])
axes[0].set(title='Conteúdo disponível para indexação', xlabel='', ylabel='Registros')
axes[0].tick_params(axis='x', rotation=15)
status_mensal = (df.assign(mes=df['Data'].dt.to_period('M').dt.to_timestamp())
                  .groupby(['mes', 'fonte_documento']).size().unstack(fill_value=0))
status_mensal.plot.area(ax=axes[1], alpha=.8)
axes[1].set(title='Cobertura textual ao longo do tempo', xlabel='', ylabel='Registros')
plt.tight_layout()

## 4. Distribuições temporal, institucional e geográfica

Essas distribuições revelam cobertura, sazonalidade e grupos dominantes. Em RAG, maior volume implica maior probabilidade de recuperação; portanto, frequência documental não deve ser confundida com relevância substantiva.

In [ ]:
df['ano'] = df['Data'].dt.year.astype('Int64')
df['mes'] = df['Data'].dt.to_period('M').dt.to_timestamp()
df['dia_semana'] = df['Data'].dt.day_name()
mensal = df.groupby('mes').size().rename('discursos').reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
sns.lineplot(data=mensal, x='mes', y='discursos', marker='o', ax=axes[0])
axes[0].set(title='Discursos por mês', xlabel='', ylabel='Discursos')
sns.boxplot(data=mensal.assign(ano=mensal['mes'].dt.year), x='ano', y='discursos', ax=axes[1])
axes[1].set(title='Distribuição mensal por ano', xlabel='Ano', ylabel='Discursos/mês')
plt.tight_layout()
display(mensal['discursos'].describe().round(2).to_frame())

In [ ]:
def ranking(coluna: str, n: int = 15) -> pd.DataFrame:
    serie = df[coluna].replace('', 'Não informado').value_counts().head(n)
    return serie.rename('discursos').rename_axis(coluna).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, coluna, titulo in zip(
    axes, ['NomeAutor', 'Partido', 'UF'],
    ['Autores mais frequentes', 'Partidos mais frequentes', 'UFs mais frequentes']
):
    dados = ranking(coluna, 12).sort_values('discursos')
    sns.barplot(data=dados, x='discursos', y=coluna, ax=ax)
    ax.set(title=titulo, xlabel='Discursos', ylabel='')
plt.tight_layout()

In [ ]:
partidos_top = ranking('Partido', 10)['Partido'].tolist()
matriz_partido_ano = pd.crosstab(df['Partido'], df['ano']).reindex(partidos_top).fillna(0)
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.heatmap(matriz_partido_ano, annot=True, fmt='.0f', cmap='Blues', ax=ax)
ax.set(title='Volume dos principais partidos por ano', xlabel='Ano', ylabel='Partido')
plt.tight_layout()

## 5. Qualidade textual e adequação para indexação

Medidas de palavras, caracteres e linhas identificam documentos muito curtos, cauda longa e possíveis artefatos. A estimativa de tokens é deliberadamente aproximada; o cálculo definitivo deverá usar o tokenizador do modelo de embeddings escolhido.

In [ ]:
texto = df['documento_rag'].fillna('').astype(str)
df['n_caracteres'] = texto.str.len()
df['n_palavras'] = texto.str.count(r'\S+')
df['n_linhas'] = texto.str.count(r'\n') + texto.ne('').astype(int)
df['tokens_estimados'] = np.ceil(df['n_caracteres'] / 4).astype(int)
df['razao_caracteres_palavra'] = np.where(df['n_palavras'] > 0, df['n_caracteres'] / df['n_palavras'], np.nan)

quantis = df.loc[df['n_palavras'] > 0, ['n_caracteres', 'n_palavras', 'n_linhas', 'tokens_estimados']].quantile(
    [0, .01, .05, .25, .5, .75, .9, .95, .99, 1]
).round(1)
display(quantis)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df.loc[df['n_palavras'] > 0, 'n_palavras'], bins=80, ax=axes[0])
axes[0].set(xscale='log', title='Comprimento dos documentos (escala log)', xlabel='Palavras', ylabel='Documentos')
sns.ecdfplot(data=df.query('n_palavras > 0'), x='tokens_estimados', hue='fonte_documento', ax=axes[1])
axes[1].set(xscale='log', title='Distribuição acumulada do tamanho', xlabel='Tokens estimados', ylabel='Proporção acumulada')
plt.tight_layout()

In [ ]:
padroes = {
    'vazio': texto.eq(''),
    'menos_de_50_palavras': df['n_palavras'].between(1, 49),
    'mais_de_10_mil_palavras': df['n_palavras'].gt(10_000),
    'possui_html': texto.str.contains(r'<[^>]+>', regex=True, na=False),
    'possui_url': texto.str.contains(r'https?://|www\.', regex=True, case=False, na=False),
    'espaços_repetidos': texto.str.contains(r' {3,}', regex=True, na=False),
    'caractere_substituição': texto.str.contains('�', regex=False, na=False),
}
qualidade_textual = pd.DataFrame({
    nome: {'registros': mascara.sum(), 'percentual': 100 * mascara.mean()}
    for nome, mascara in padroes.items()
}).T.round(2).sort_values('registros', ascending=False)
display(qualidade_textual)

### 5.1 Duplicidade exata e conteúdo repetido

Documentos idênticos podem gerar chunks redundantes, dominar os vizinhos mais próximos e reduzir a diversidade dos resultados. A normalização abaixo é conservadora e serve apenas para diagnosticar duplicidade exata após diferenças triviais de caixa e espaços.

In [ ]:
def normalizar_para_hash(valor: str) -> str:
    valor = unicodedata.normalize('NFKC', valor).casefold()
    return re.sub(r'\s+', ' ', valor).strip()

normalizado = texto.map(normalizar_para_hash)
df['hash_texto'] = normalizado.map(lambda x: hashlib.sha256(x.encode('utf-8')).hexdigest() if x else '')
duplicados_texto = df.loc[df['hash_texto'].ne('') & df['hash_texto'].duplicated(keep=False)]
grupos_duplicados = duplicados_texto.groupby('hash_texto').size().sort_values(ascending=False)

duplicidade = pd.Series({
    'registros em grupos duplicados': len(duplicados_texto),
    'grupos de texto duplicado': len(grupos_duplicados),
    'cópias excedentes': int((grupos_duplicados - 1).sum()),
    'percentual de cópias excedentes': round(100 * (grupos_duplicados - 1).sum() / len(df), 2),
})
display(duplicidade.to_frame('valor'))
display(grupos_duplicados.head(10).rename('ocorrências').to_frame())

## 6. Vocabulário e variação lexical

A frequência de termos ajuda a encontrar ruído, fórmulas regimentais e vocabulário dominante. Não substitui análise temática. O ajuste limita-se aos documentos integrais e usa `min_df` para evitar que erros raros dominem o vocabulário.

In [ ]:
STOPWORDS_PT = {
    'a','ao','aos','aquela','aquele','aqueles','as','com','como','da','das','de','dela','dele','do','dos',
    'e','ela','ele','eles','em','entre','era','essa','esse','esta','este','eu','foi','há','isso','isto',
    'já','mais','mas','me','mesmo','meu','minha','muito','na','nas','não','no','nos','nós','o','os','ou',
    'para','pela','pelo','por','porque','que','quem','se','sem','ser','seu','sua','também','tem','ter',
    'um','uma','você','sr','sra','senhor','senhora','presidente'
}
corpus_integral = df.loc[df['fonte_documento'].eq('texto_integral'), 'documento_rag']
vectorizer = CountVectorizer(
    lowercase=True, strip_accents='unicode', stop_words=sorted(STOPWORDS_PT),
    min_df=10, max_df=.98, token_pattern=r'(?u)\b[a-zA-ZÀ-ÿ]{3,}\b',
)
X = vectorizer.fit_transform(corpus_integral)
frequencias = np.asarray(X.sum(axis=0)).ravel()
termos = pd.DataFrame({'termo': vectorizer.get_feature_names_out(), 'frequencia': frequencias})
top_termos = termos.nlargest(30, 'frequencia').sort_values('frequencia')

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=top_termos, x='frequencia', y='termo', ax=ax)
ax.set(title='Termos mais frequentes nos textos integrais', xlabel='Ocorrências', ylabel='')
plt.tight_layout()

In [ ]:
vetor_bi = CountVectorizer(
    lowercase=True, strip_accents='unicode', stop_words=sorted(STOPWORDS_PT),
    ngram_range=(2, 2), min_df=20, max_features=5_000,
)
bigramas = vetor_bi.fit_transform(corpus_integral)
top_bi = pd.DataFrame({
    'bigrama': vetor_bi.get_feature_names_out(),
    'frequencia': np.asarray(bigramas.sum(axis=0)).ravel(),
}).nlargest(25, 'frequencia').sort_values('frequencia')
fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=top_bi, x='frequencia', y='bigrama', ax=ax)
ax.set(title='Bigramas mais frequentes', xlabel='Ocorrências', ylabel='')
plt.tight_layout()

### 6.1 Mudança lexical entre anos

A distância de Jensen–Shannon compara distribuições de termos e varia de 0 (iguais) a 1 com logaritmo de base 2 (muito distintas). Mudanças elevadas sugerem que consultas, avaliações e amostras do RAG devem cobrir diferentes períodos.

In [ ]:
indices_integrais = df.index[df['fonte_documento'].eq('texto_integral')]
anos_integral = df.loc[indices_integrais, 'ano'].astype(str).to_numpy()
anos_validos = sorted(pd.unique(anos_integral))
distribuicoes = {}
for ano in anos_validos:
    contagens = np.asarray(X[anos_integral == ano].sum(axis=0)).ravel().astype(float) + 1e-12
    distribuicoes[ano] = contagens / contagens.sum()

js = pd.DataFrame(index=anos_validos, columns=anos_validos, dtype=float)
for ano_a in anos_validos:
    for ano_b in anos_validos:
        js.loc[ano_a, ano_b] = jensenshannon(distribuicoes[ano_a], distribuicoes[ano_b], base=2)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(js, annot=True, fmt='.3f', cmap='mako', vmin=0, ax=ax)
ax.set(title='Distância lexical de Jensen–Shannon entre anos', xlabel='Ano', ylabel='Ano')
plt.tight_layout()

## 7. Simulação de chunking e custo de indexação

A simulação usa 512 tokens por chunk e sobreposição de 64 tokens. É uma estimativa para planejamento; a implementação deverá segmentar por unidades semânticas e medir tokens com o tokenizador do modelo escolhido. Resumos de fallback devem permanecer em uma coleção ou camada identificável, pois não têm a mesma granularidade dos discursos integrais.

In [ ]:
CHUNK_TOKENS = 512
OVERLAP_TOKENS = 64
PASSO = CHUNK_TOKENS - OVERLAP_TOKENS
df['chunks_estimados'] = np.where(
    df['tokens_estimados'].eq(0),
    0,
    np.maximum(1, np.ceil((df['tokens_estimados'] - OVERLAP_TOKENS) / PASSO)),
).astype(int)

resumo_chunks = (df.groupby('fonte_documento')
                  .agg(documentos=('CodigoPronunciamento', 'size'),
                       tokens_estimados=('tokens_estimados', 'sum'),
                       chunks_estimados=('chunks_estimados', 'sum'),
                       mediana_chunks=('chunks_estimados', 'median'),
                       p95_chunks=('chunks_estimados', lambda s: s.quantile(.95)),
                       máximo_chunks=('chunks_estimados', 'max')))
display(resumo_chunks)

In [ ]:
parametros = []
for tamanho in [256, 512, 768, 1024]:
    sobreposicao = tamanho // 8
    passo = tamanho - sobreposicao
    chunks = np.where(df['tokens_estimados'].eq(0), 0, np.maximum(1, np.ceil((df['tokens_estimados'] - sobreposicao) / passo)))
    parametros.append({
        'tamanho_chunk': tamanho, 'sobreposição': sobreposicao,
        'chunks_totais': int(chunks.sum()), 'chunks_por_documento': round(chunks.mean(), 2),
        'documentos_em_um_chunk_pct': round(100 * np.mean(chunks == 1), 2),
    })
comparacao_chunking = pd.DataFrame(parametros)
display(comparacao_chunking)
fig, ax = plt.subplots()
sns.barplot(data=comparacao_chunking, x='tamanho_chunk', y='chunks_totais', ax=ax)
ax.set(title='Impacto do tamanho do chunk no índice', xlabel='Tokens por chunk', ylabel='Chunks estimados')
plt.tight_layout()

## 8. Metadados e riscos para recuperação

Filtros por data, autor, partido, UF e tipo de uso podem elevar precisão e explicabilidade. A tabela mede a disponibilidade desses campos somente entre documentos indexáveis.

In [ ]:
indexaveis = df[df['fonte_documento'].ne('indisponível')].copy()
campos_filtro = ['CodigoPronunciamento', 'Data', 'NomeAutor', 'Partido', 'UF', 'TipoUsoPalavra.Descricao', 'Indexacao']
cobertura_metadados = []
for coluna in campos_filtro:
    preenchido = indexaveis[coluna].notna() & indexaveis[coluna].astype(str).str.strip().ne('')
    cobertura_metadados.append({
        'campo': coluna, 'preenchidos': int(preenchido.sum()),
        'cobertura_pct': round(100 * preenchido.mean(), 2),
        'valores_unicos': indexaveis.loc[preenchido, coluna].nunique(),
    })
cobertura_metadados = pd.DataFrame(cobertura_metadados).sort_values('cobertura_pct')
display(cobertura_metadados)
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(data=cobertura_metadados, x='cobertura_pct', y='campo', ax=ax)
ax.set(xlim=(0, 105), title='Cobertura dos metadados úteis para filtros', xlabel='Cobertura (%)', ylabel='')
plt.tight_layout()

In [ ]:
concentracao = {}
for coluna in ['NomeAutor', 'Partido', 'UF']:
    proporcoes = indexaveis[coluna].replace('', 'Não informado').value_counts(normalize=True)
    concentracao[coluna] = {
        'participação_top_1_pct': 100 * proporcoes.iloc[0],
        'participação_top_10_pct': 100 * proporcoes.iloc[:10].sum(),
        'HHI': float((proporcoes ** 2).sum()),
    }
display(pd.DataFrame(concentracao).T.round(3))

## 9. Síntese automática de prontidão para RAG

O quadro final consolida indicadores observáveis. Os limiares são heurísticos e devem orientar investigação, não funcionar como certificado de qualidade. A qualidade real da recuperação exige um conjunto de perguntas, documentos relevantes anotados e métricas como Recall@k, MRR e nDCG.

In [ ]:
pct_indexavel = 100 * indexaveis.shape[0] / len(df)
pct_integral = 100 * df['fonte_documento'].eq('texto_integral').mean()
pct_vazios = 100 * df['fonte_documento'].eq('indisponível').mean()
pct_copias = 100 * (grupos_duplicados - 1).sum() / len(df) if len(grupos_duplicados) else 0
pct_curto = 100 * df['n_palavras'].between(1, 49).mean()
cobertura_filtros = cobertura_metadados.set_index('campo')['cobertura_pct'].min()

scorecard = pd.DataFrame([
    ['Documentos indexáveis', pct_indexavel, '>= 99%', pct_indexavel >= 99],
    ['Cobertura de texto integral', pct_integral, '>= 90%', pct_integral >= 90],
    ['Registros sem conteúdo', pct_vazios, '< 1%', pct_vazios < 1],
    ['Cópias exatas excedentes', pct_copias, '< 2%', pct_copias < 2],
    ['Documentos com menos de 50 palavras', pct_curto, '< 10%', pct_curto < 10],
    ['Menor cobertura entre filtros', cobertura_filtros, '>= 90%', cobertura_filtros >= 90],
], columns=['indicador', 'valor_pct', 'referência', 'atende'])
scorecard['valor_pct'] = scorecard['valor_pct'].round(2)
scorecard['situação'] = scorecard['atende'].map({True: 'Adequado', False: 'Requer tratamento'})
display(scorecard.drop(columns='atende'))

In [ ]:
recomendacoes = [
    'Preservar CodigoPronunciamento, data, autor, partido, UF e proveniência em cada chunk.',
    'Indexar texto integral e resumo de fallback com marcadores distintos; não apresentá-los como equivalentes.',
    'Excluir da indexação os registros sem texto e sem resumo, mantendo-os no relatório de cobertura.',
    'Segmentar por parágrafos ou sentenças antes de aplicar limite de tokens e manter pequena sobreposição.',
    'Deduplicar apenas após inspeção: repetições podem ser registros legislativos legítimos.',
    'Combinar recuperação semântica com busca lexical e filtros de metadados.',
    'Criar um conjunto de avaliação estratificado por ano, partido, UF, tamanho e fonte do documento.',
    'Avaliar recuperação (Recall@k, MRR, nDCG), fidelidade da resposta e correção das citações separadamente.',
]
display(Markdown('### Recomendações para o pipeline\n' + '\n'.join(f'- {r}' for r in recomendacoes)))

## 10. Exportação opcional de artefatos

A exportação fica desativada por padrão. Quando habilitada, grava somente tabelas derivadas em `texto-Latex/dados/`; nunca modifica o corpus original. As figuras podem ser salvas seletivamente após definição do padrão visual da dissertação.

In [ ]:
EXPORTAR = False
if EXPORTAR:
    raiz_dissertacao = next(
        p for p in [Path.cwd() / '13-dissertacao', Path.cwd().parent, Path.cwd()]
        if (p / 'texto-Latex').exists()
    )
    destino = raiz_dissertacao / 'texto-Latex' / 'dados'
    destino.mkdir(parents=True, exist_ok=True)
    visao_colunas.to_csv(destino / 'auditoria_colunas.csv', encoding='utf-8', index=True)
    cobertura.to_csv(destino / 'cobertura_textual.csv', encoding='utf-8', index=True)
    comparacao_chunking.to_csv(destino / 'estimativa_chunking.csv', encoding='utf-8', index=False)
    scorecard.drop(columns='atende').to_csv(destino / 'prontidao_rag.csv', encoding='utf-8', index=False)
    print(f'Artefatos exportados para {destino}')
else:
    print('Exportação desativada; nenhum arquivo foi criado.')

## 11. Próximas etapas

1. revisar amostras de documentos curtos, longos e duplicados;
2. definir o modelo de embeddings e calcular tokens com seu tokenizador;
3. comparar estratégias de chunking semântico e por tamanho;
4. criar perguntas de avaliação com julgamentos de relevância;
5. testar busca lexical, vetorial e híbrida com os mesmos casos;
6. registrar versões, parâmetros, métricas e decisões metodológicas na dissertação.